# Window Embeddings — Time-Aware RAG Demo
Computes sliding-window embeddings for all 12,695 CAQA passages using your fine-tuned Contriever.  
**Runtime**: ~5–10 min on A100.  
**Output**: `window_state.pt` (~340 MB) uploaded directly to your HF Space.

Run all cells top-to-bottom. No manual steps needed after Cell 2 (HF login).

In [ ]:
# Cell 1 — Install dependencies
!pip install -q transformers torch datasets faiss-cpu huggingface_hub nltk tqdm

In [ ]:
# Cell 2 — Login to HuggingFace (needed to upload to your Space)
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# Cell 3 — Verify GPU
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
# Cell 4 — Load CAQA passages (exact same order as chroniclingqa_eval.py)
import re, json
from datasets import load_dataset
from tqdm import tqdm

def _pick_field(ex, field_names):
    for field in field_names:
        if field in ex and ex[field] is not None:
            txt = str(ex[field]).strip()
            if txt:
                return txt
    return None

print('Loading ChroniclingAmericaQA validation split...')
caqa = load_dataset('Bhawna/ChroniclingAmericaQA', split='validation')

ca_passage_text_to_id = {}
ca_passages_list = []

for ex in tqdm(caqa, desc='Building passage list'):
    question = _pick_field(ex, ['question', 'query', 'input', 'prompt', 'text'])
    passage  = _pick_field(ex, ['positive_passage', 'passage', 'context', 'document',
                                 'evidence', 'target', 'passage_text', 'ctx'])
    if question is None or passage is None:
        continue
    if passage not in ca_passage_text_to_id:
        pid = len(ca_passages_list)
        ca_passage_text_to_id[passage] = pid
        ca_passages_list.append(passage)

print(f'Total unique passages: {len(ca_passages_list)}')
assert len(ca_passages_list) == 12695, f'Expected 12695, got {len(ca_passages_list)} — dataset revision mismatch!'

In [ ]:
# Cell 5 — Load fine-tuned model from HF Hub
from transformers import AutoModel, AutoTokenizer

MODEL_HUB_ID  = 'manojarulmurugan/time-aware-contriever'
BASE_MODEL_ID = 'facebook/contriever-msmarco'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'Loading tokenizer from {BASE_MODEL_ID}...')
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)

print(f'Loading model from {MODEL_HUB_ID}...')
model = AutoModel.from_pretrained(MODEL_HUB_ID).to(DEVICE).eval()
print(f'Model loaded on {DEVICE}')

In [ ]:
# Cell 6 — Precompute window embeddings (the only missing artifact)
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

WINDOW_SIZE   = 3
WINDOW_STRIDE = 1
BATCH_SIZE    = 256   # A100 can handle large batches
MAX_LEN       = 128
AMP_DTYPE     = torch.float16 if DEVICE == 'cuda' else torch.float32

def mean_pooling(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    return torch.sum(last_hidden_state * mask, 1) / torch.clamp(mask.sum(1), min=1e-9)

# Step 1: Sentence-split every passage into overlapping windows
print('Step 1/2: Splitting passages into windows...')
all_window_texts = []
doc_window_map   = {}   # {str(pid): (start_idx, count)}
current_idx = 0

for pid, text in enumerate(tqdm(ca_passages_list, desc='Windowing')):
    try:
        snts = nltk.sent_tokenize(text or '')
    except Exception:
        snts = re.split(r'(?<=[.!?])\s+', text or '')
    snts = [s.strip() for s in snts if s.strip()]

    if not snts:
        windows = ['']
    elif len(snts) <= WINDOW_SIZE:
        windows = [' '.join(snts)]
    else:
        windows = [' '.join(snts[i:i+WINDOW_SIZE])
                   for i in range(0, len(snts) - WINDOW_SIZE + 1, WINDOW_STRIDE)]

    doc_window_map[str(pid)] = (current_idx, len(windows))
    all_window_texts.extend(windows)
    current_idx += len(windows)

print(f'Total windows: {len(all_window_texts)}')

# Step 2: Encode all windows in batches on GPU
print('Step 2/2: Encoding windows on GPU...')
all_embs = []

for i in tqdm(range(0, len(all_window_texts), BATCH_SIZE), desc='Encoding'):
    batch = all_window_texts[i:i+BATCH_SIZE]
    toks  = tokenizer(batch, padding=True, truncation=True,
                      max_length=MAX_LEN, return_tensors='pt').to(DEVICE)
    with torch.no_grad(), torch.autocast(DEVICE, dtype=AMP_DTYPE, enabled=(DEVICE=='cuda')):
        out    = model(**toks)
        pooled = mean_pooling(out.last_hidden_state, toks['attention_mask'])
        pooled = torch.nn.functional.normalize(pooled, p=2, dim=1)
    all_embs.append(pooled.cpu().float())

window_emb_tensor = torch.cat(all_embs, dim=0)
print(f'window_emb_tensor shape: {window_emb_tensor.shape}')   # (~110k, 768)
print(f'doc_window_map entries: {len(doc_window_map)}')

In [ ]:
# Cell 7 — Save window_state.pt locally and also save passages.json
import os

print('Saving window_state.pt...')
torch.save({'window_emb_tensor': window_emb_tensor, 'doc_window_map': doc_window_map},
           'window_state.pt')

print('Saving passages.json...')
with open('passages.json', 'w') as f:
    json.dump(ca_passages_list, f)

ws_mb = os.path.getsize('window_state.pt') / 1e6
pa_mb = os.path.getsize('passages.json') / 1e6
print(f'window_state.pt: {ws_mb:.1f} MB')
print(f'passages.json:   {pa_mb:.1f} MB')
assert ws_mb > 100, 'window_state.pt seems too small — something went wrong!'

In [ ]:
# Cell 8 — Upload both files directly to the HF Space
from huggingface_hub import HfApi
api = HfApi()

SPACE_ID = 'manojarulmurugan/time-aware-rag'

print('Uploading window_state.pt to HF Space (this may take 2-3 min)...')
api.upload_file(
    path_or_fileobj='window_state.pt',
    path_in_repo='demo_data/window_state.pt',
    repo_id=SPACE_ID,
    repo_type='space',
    commit_message='Add full 12,695-passage window embeddings (A100 computed)',
)
print('window_state.pt uploaded.')

print('Uploading passages.json to HF Space...')
api.upload_file(
    path_or_fileobj='passages.json',
    path_in_repo='demo_data/passages.json',
    repo_id=SPACE_ID,
    repo_type='space',
    commit_message='Add full 12,695-passage text corpus',
)
print('passages.json uploaded.')
print('\nDone! The HF Space will now rebuild with the full corpus.')